# Project 4: NLP & Sentiment Analysis
# DecodeLabs Industrial Training Kit

In [ ]:
import re
import nltk
import numpy as np
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.datasets import load_files
import scipy.sparse as sp

# Download required NLTK resources

In [ ]:
for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger',
                 'punkt_tab', 'averaged_perceptron_tagger_eng']:
    nltk.download(resource, quiet=True)

# STEP 1 — TEXT PRE-PROCESSING PIPELINE

In [ ]:
def get_wordnet_pos(treebank_tag):
    """
    Map Penn Treebank POS tags → WordNet POS tags.
    Mandatory for accurate POS-guided lemmatization (as per blueprint).
    """
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
      return wordnet.NOUN  # Default assumption


def build_custom_stopwords():
    """
    Engineering Rule: Explicitly EXCLUDE negations from the NLTK stopword list.
    Removing 'not' would flip sentiment (e.g., 'not happy' → 'happy').
    """
    negations = {
        "no", "not", "nor", "neither", "never", "nobody", "nothing",
        "nowhere", "isn't", "aren't", "wasn't", "weren't", "haven't",
        "hasn't", "hadn't", "won't", "wouldn't", "don't", "doesn't",
        "didn't", "can't", "couldn't", "shouldn't", "mightn't", "mustn't"
    }
    base_stopwords = set(stopwords.words('english'))
    # Set-based union operation: remove negations from the default list
    custom_stopwords = base_stopwords - negations
    return custom_stopwords


def preprocess_text(text, lemmatizer, custom_stopwords):
    """
    Full pre-processing pipeline:
      1. Character Normalization  — strip HTML tags, lowercase, remove punctuation
      2. Tokenization             — split into individual word tokens
      3. Stop-Word Removal        — filter with negation-safe custom stopword set
      4. POS-Guided Lemmatization — reduce to true root form using WordNet tags
    """
    # --- Stage 1: Character Normalization ---
    text = re.sub(r'<[^>]+>', ' ', text)        # Remove HTML tags (e.g., <br>)
    text = text.lower()                          # Lowercase everything
    text = re.sub(r'[^a-z\s]', ' ', text)       # Remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()     # Collapse extra whitespace

    # --- Stage 2: Tokenization ---
    tokens = word_tokenize(text)

    # --- Stage 3: Stop-Word Removal (negation-safe) ---
    tokens = [t for t in tokens if t not in custom_stopwords and len(t) > 1]

    # --- Stage 4: POS-Guided Lemmatization (WordNetLemmatizer) ---
    pos_tags = pos_tag(tokens)                   # Get Treebank POS tags
    lemmatized = [
        lemmatizer.lemmatize(token, get_wordnet_pos(tag))
        for token, tag in pos_tags
    ]

    return ' '.join(lemmatized)

# STEP 2 — SAMPLE DATASET (Product Reviews)

In [ ]:
def get_sample_data():
    """
    Built-in demo dataset of labeled product reviews.
    Labels: 1 = Positive, 0 = Negative
    """
    reviews = [
        # Positive reviews
        ("This product is absolutely amazing! Best purchase I've ever made.", 1),
        ("Excellent quality, fast delivery. Highly recommend to everyone.", 1),
        ("I love this item. Works perfectly and looks great.", 1),
        ("Outstanding performance. Worth every penny spent.", 1),
        ("Really good product, very satisfied with the results.", 1),
        ("Fantastic build quality. Not cheap at all, very premium feel.", 1),
        ("The best on the market. I am not disappointed at all.", 1),
        ("Great value for money. My family loves it.", 1),
        ("Superb quality and packaging. Will definitely buy again.", 1),
        ("Works exactly as described. Very happy with this purchase.", 1),
        ("Incredible product! Exceeded all my expectations completely.", 1),
        ("Top notch quality. Very well made and durable.", 1),
        ("This is wonderful! Not what I expected—even better!", 1),
        ("Perfect fit, great material, arrived quickly. Love it.", 1),
        ("Very satisfied customer. Works flawlessly every single time.", 1),

        # Negative reviews
        ("This product is absolutely terrible. Complete waste of money.", 0),
        ("Worst purchase ever. It broke after just two days of use.", 0),
        ("I am not happy with this at all. Very poor quality.", 0),
        ("Disappointed. It does not work as advertised, very misleading.", 0),
        ("Cheap plastic junk. Would not recommend this to anyone.", 0),
        ("Stopped working after one week. Extremely poor build quality.", 0),
        ("Not good at all. Returned it immediately for a refund.", 0),
        ("Horrible product. Customer service was no help either.", 0),
        ("Total scam. Nothing like the pictures shown online.", 0),
        ("Very bad experience. The item arrived broken and unusable.", 0),
        ("Awful quality. It is not worth the price they are charging.", 0),
        ("Do not buy this. It is a complete disappointment overall.", 0),
        ("I couldn't believe how bad this product is. Waste of time.", 0),
        ("Terrible smell, bad materials. Not suitable for any use.", 0),
        ("Defective product. Doesn't match the description at all.", 0),
    ]
    texts, labels = zip(*reviews)
    return list(texts), list(labels)

# STEP 3 — VECTORIZATION (TF-IDF with N-Grams)

In [ ]:
def build_tfidf_vectorizer(preprocessed_texts):
    """
    TF-IDF Vectorizer with:
      - Unigrams + Bigrams (1,2) to capture negated phrases like 'not good'
      - max_features=10000 to prevent dimensionality explosion
      - min_df=2 to exclude rare typos / one-off tokens
      - sublinear_tf=True for log-scaled term frequency (better for text)
    Output is automatically stored as SciPy CSR Sparse Matrix (memory optimal).
    """
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),       # Unigrams and Bigrams
        max_features=10_000,      # Cap vocabulary to control dimensionality
        min_df=2,                 # Ignore terms appearing in fewer than 2 docs
        sublinear_tf=True         # Apply log(1 + tf) for smoother frequency
    )
    X = vectorizer.fit_transform(preprocessed_texts)  # Returns CSR sparse matrix
    return vectorizer, X

# STEP 4 — MODEL TRAINING & INFERENCE

In [ ]:
def select_classifier(labels):
    """
    Choose the appropriate Naive Bayes variant based on class balance:
      - MultinomialNB  → for balanced datasets (~50/50 split)
      - ComplementNB   → for imbalanced datasets (corrects extreme class bias)
    """
    pos_ratio = sum(labels) / len(labels)
    is_balanced = 0.35 <= pos_ratio <= 0.65

    if is_balanced:
        print(f"  Dataset is balanced ({pos_ratio:.0%} positive). Using MultinomialNB.")
        return MultinomialNB(alpha=1.0)   # alpha=1.0 → Laplace Smoothing
    else:
        print(f"  Dataset is imbalanced ({pos_ratio:.0%} positive). Using ComplementNB.")
        return ComplementNB(alpha=1.0)    # Handles skewed class distributions


def train_and_evaluate(X, labels):
    """
    Split data, train the selected classifier, and print evaluation metrics.
    """
    y = np.array(labels)

    # Stratified split preserves class ratio in both train and test sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"\n  Training samples : {X_train.shape[0]}")
    print(f"  Test samples     : {X_test.shape[0]}")
    print(f"  Feature dims     : {X_train.shape[1]} (CSR sparse)")

    # Select and train classifier
    clf = select_classifier(labels)
    clf.fit(X_train, y_train)

    # Evaluate
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"\n  Accuracy: {acc * 100:.2f}%")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=["Negative", "Positive"]))
    return clf

# STEP 5 — PREDICTION INTERFACE

In [ ]:
def predict_sentiment(texts, vectorizer, clf, lemmatizer, custom_stopwords):
    """
    Run the full pipeline on new, unseen review texts and return predictions.
    """
    preprocessed = [
        preprocess_text(t, lemmatizer, custom_stopwords) for t in texts
    ]
    X_new = vectorizer.transform(preprocessed)   # CSR sparse, same feature space
    predictions = clf.predict(X_new)
    probabilities = clf.predict_proba(X_new)

    print("\n" + "=" * 60)
    print("  LIVE SENTIMENT PREDICTIONS")
    print("=" * 60)
    for text, pred, prob in zip(texts, predictions, probabilities):
        label = "POSITIVE ✓" if pred == 1 else "NEGATIVE ✗"
        confidence = max(prob) * 100
        print(f"\n  Review     : {text[:70]}...")
        print(f"  Sentiment  : {label}  |  Confidence: {confidence:.1f}%")
    print("=" * 60)

# MAIN — ORCHESTRATE THE FULL PIPELINE

In [ ]:
def main():
    print("=" * 60)
    print("  PROJECT 4: NLP & SENTIMENT ANALYSIS ENGINE")
    print("  DecodeLabs Industrial Training Kit ")
    print("=" * 60)

    # --- Initialize NLP tools ---
    lemmatizer = WordNetLemmatizer()
    custom_stopwords = build_custom_stopwords()
    print(f"\n  [1/5] Stop-words loaded. Negations preserved.")
    print(f"        Vocabulary size: {len(custom_stopwords)} words")

    # --- Load dataset ---
    texts, labels = get_sample_data()
    print(f"\n  [2/5] Dataset loaded: {len(texts)} reviews.")

    # --- Pre-process ---
    print("\n  [3/5] Running pre-processing pipeline...")
    preprocessed = [
        preprocess_text(t, lemmatizer, custom_stopwords) for t in texts
    ]
    print("        Sample preprocessed text:")
    print(f"        IN  → {texts[0]}")
    print(f"        OUT → {preprocessed[0]}")

    # --- Vectorize ---
    print("\n  [4/5] Vectorizing with TF-IDF (Unigrams + Bigrams, CSR sparse)...")
    vectorizer, X = build_tfidf_vectorizer(preprocessed)
    print(f"        Matrix shape : {X.shape}")
    print(f"        Sparse format: {type(X).__name__} (SciPy CSR)")
    print(f"        Non-zero cells: {X.nnz} / {X.shape[0] * X.shape[1]}")

    # --- Train & Evaluate ---
    print("\n  [5/5] Training classifier & evaluating...")
    clf = train_and_evaluate(X, labels)

    # --- Live Predictions ---
    new_reviews = [
        "This product is not good at all. I am very disappointed with quality.",
        "Absolutely love this! Best thing I have bought in a long time.",
        "Terrible experience. It stopped working after just one day of use.",
        "The quality is outstanding and delivery was surprisingly fast.",
    ]
    predict_sentiment(new_reviews, vectorizer, clf, lemmatizer, custom_stopwords)


if __name__ == "__main__":
    main()

  PROJECT 4: NLP & SENTIMENT ANALYSIS ENGINE
  DecodeLabs Industrial Training Kit 

  [1/5] Stop-words loaded. Negations preserved.
        Vocabulary size: 179 words

  [2/5] Dataset loaded: 30 reviews.

  [3/5] Running pre-processing pipeline...
        Sample preprocessed text:
        IN  → This product is absolutely amazing! Best purchase I've ever made.
        OUT → product absolutely amazing best purchase ever make

  [4/5] Vectorizing with TF-IDF (Unigrams + Bigrams, CSR sparse)...
        Matrix shape : (30, 37)
        Sparse format: csr_matrix (SciPy CSR)
        Non-zero cells: 100 / 1110

  [5/5] Training classifier & evaluating...

  Training samples : 24
  Test samples     : 6
  Feature dims     : 37 (CSR sparse)
  Dataset is balanced (50% positive). Using MultinomialNB.

  Accuracy: 66.67%

  Classification Report:
              precision    recall  f1-score   support

    Negative       1.00      0.33      0.50         3
    Positive       0.60      1.00      0.75    